# LifeMate — train tiếp hynt từ output đã hoàn thành

**Add Input output cũ + dataset audio gốc → GPU + Internet → Save Version → Save & Run All.**
`PREVIOUS_OUTPUT` đã điền theo báo cáo trước của bạn; chỉ đổi nếu Kaggle mount bằng đường dẫn khác.
Không cần bấm Run All trong editor trước. Quick Save chỉ lưu, không train thêm.

Mặc định: tối đa **5 epoch thêm**, LR đầu phase = **25% LR cấu hình run gốc**, patience **3**.
Nếu run cũ xong epoch 5, phase mới chạy tối đa epoch 6–10. Đây là lựa chọn thử nghiệm, không đảm bảo tăng điểm.
Tiếp tục **weights + optimizer + scaler + RNG từ last.pt**; giữ **best.pt cũ** và điểm/predictions mọi epoch.
Không quay lại weights epoch tốt nhất: best dùng chọn model, last dùng tiếp nối trạng thái train.
Không chạy lại baseline/preflight. Setup vẫn tải pretrained để dựng model/kiểm tra kiến trúc trước khi nạp last.pt.

Đây là **phase mới**, không phải kéo dài nguyên lịch train cũ: reset patience đúng một lần,
khởi động lịch cosine LR mới từ mức trên, vẫn giữ bộ đếm updates và lịch encoder từ run cũ.
Best chọn bằng EM tăng, hòa EM thì WER giảm. Output ghi riêng, input cũ chỉ đọc.
Nếu hết budget: Save output, Add Input output continuation mới nhất, giữ EXTRA_EPOCHS/NEW_PATIENCE,
đổi OUTPUT_DIR mới; phase đang dở giữ lịch LR và patience, không cộng lại 5 epoch.
Không dùng notebook này để mở thêm phase sau khi phase continuation đã completed.

Điểm vẫn là validation để chọn checkpoint, không phải test độc lập; giữ output private.


## 1. Điền output cũ và cấu hình phase mới


In [ ]:
from pathlib import Path
import json, hashlib

# Add Input: output notebook train cũ VÀ dataset audio gốc.
PREVIOUS_OUTPUT = '/kaggle/input/notebooks/jamel27/zaic-lifemate-hyntzipformer-finetune/lifemate_hynt_finetune_fresh_5epoch_em'  # Đã điền output gốc; đổi nếu dùng version khác.
DATASET_ROOT = Path('/kaggle/input/datasets/jamel27/zalo-aichallange-lifemate-dataset')
EXTRA_EPOCHS = 5      # Tối đa 5 epoch THÊM, không phải tổng epoch từ pretrained.
NEW_PATIENCE = 3     # EM tăng, hoặc EM bằng và WER giảm, mới được xem là cải thiện.
LR_FACTOR = 0.25     # LR đầu phase = LR cấu hình run gốc × hệ số này.
MAX_HOURS = 9.0      # Hết budget: lưu last.pt và chấm valid, có thể tiếp tục ở phiên sau.
OUTPUT_DIR = Path('/kaggle/working/lifemate_hynt_continue_stage1_em')  # Output mới, không ghi đè run cũ.
WORK_DIR = Path('/kaggle/temp/lifemate_hynt_continue')

if not PREVIOUS_OUTPUT.strip():
    raise ValueError('Điền PREVIOUS_OUTPUT trước khi Save & Run All')
if not isinstance(EXTRA_EPOCHS,int) or EXTRA_EPOCHS<1 or not isinstance(NEW_PATIENCE,int) or NEW_PATIENCE<1:
    raise ValueError('EXTRA_EPOCHS và NEW_PATIENCE phải là số nguyên dương')
if not 0<LR_FACTOR<=1:
    raise ValueError('LR_FACTOR phải trong (0,1]')
RESUME_DIR=Path(PREVIOUS_OUTPUT).expanduser()
if not RESUME_DIR.is_dir():
    raise FileNotFoundError(f'Chưa thấy output cũ: {RESUME_DIR}. Add Input → Notebook Output của đúng model, rồi kiểm tra PREVIOUS_OUTPUT có khớp đường dẫn mount không.')
RESUME_DIR=RESUME_DIR.resolve(strict=True)
if OUTPUT_DIR.resolve()==RESUME_DIR or RESUME_DIR in OUTPUT_DIR.resolve().parents:
    raise ValueError('Output phải là thư mục mới nằm ngoài input cũ')
for name in ('last.pt','best.pt','run_identity.json','split_summary.json','valid.jsonl',
             'baseline_metrics.json','baseline_predictions.jsonl','bpe.model','hynt_finetune.py','hynt_finetune_data.py'):
    if not (RESUME_DIR/name).is_file():raise FileNotFoundError(f'Thiếu {name} trong {RESUME_DIR}')
previous=json.loads((RESUME_DIR/'run_identity.json').read_text(encoding='utf-8'))
previous_config=previous['config']
if previous_config.get('selection_metric')!='normalized_exact_match_then_wer':
    raise ValueError('Chỉ tiếp tục run chọn checkpoint theo EM rồi WER')
existing_phase=RESUME_DIR/'continuation_plan.json'
if not existing_phase.exists():
    prior_state=json.loads((RESUME_DIR/'state.json').read_text(encoding='utf-8'))
    if prior_state.get('state')!='completed':
        raise ValueError('Run đầu phải completed. Run partial cũ cần resume đúng notebook cũ trước.')
    if hashlib.sha256((RESUME_DIR/'hynt_finetune.py').read_text(encoding='utf-8').encode()).hexdigest()!='7c50c9444a6fdc522d126af3a18d2039b10e212ecb7c42688711415ea482b238':
        raise ValueError('Source trainer run cũ khác template đã kiểm tra; cần kiểm tra phiên bản trước khi tiếp tục')
    if hashlib.sha256((RESUME_DIR/'hynt_finetune_data.py').read_text(encoding='utf-8').encode()).hexdigest()!='83143ddcb655739c0d9cd9892b02f3e0a3cb1eede1b13a7e8b18677a056ae115':
        raise ValueError('Source chuẩn bị dữ liệu khác phiên bản đã kiểm tra')
    LEARNING_RATE=previous_config['learning_rate']*LR_FACTOR
else:
    # Exact continuation-phase resume after time budget/session interruption.
    prior_state=json.loads((RESUME_DIR/'state.json').read_text(encoding='utf-8'))
    if prior_state.get('state')=='completed':
        raise ValueError('Continuation phase này đã completed; không tự mở lại phase hoặc reset patience')
    if hashlib.sha256((RESUME_DIR/'hynt_finetune.py').read_text(encoding='utf-8').encode()).hexdigest()!='1bbfcf9f55268ae432db490146a079e91fe91ebbc32664784ee1c57d90ee54cc':
        raise ValueError('Trainer continuation khác phiên bản đã kiểm tra')
    phase=json.loads(existing_phase.read_text(encoding='utf-8'))
    if EXTRA_EPOCHS!=phase['extra_epochs'] or NEW_PATIENCE!=phase['patience']:
        raise ValueError('Resume phase: giữ nguyên EXTRA_EPOCHS/NEW_PATIENCE của phase đã lưu')
    LEARNING_RATE=phase['learning_rate']
    print('Resume cùng continuation phase; LR_FACTOR không tính lại, patience không reset.')
SELECTION_METRIC=previous_config['selection_metric']
SEED=previous_config['seed']
VALID_FRACTION=json.loads((RESUME_DIR/'split_summary.json').read_text(encoding='utf-8'))['valid_fraction_requested']
ICEFALL_COMMIT=previous_config['icefall_commit']
GRADIENT_ACCUMULATION=previous_config['gradient_accumulation']
ENGLISH_EXTRA_FRACTION=previous_config['english_extra_fraction']
CS_CONCAT_PROBABILITY=previous_config['cs_concat_probability']
NOISE_PROBABILITY=previous_config['noise_probability']
PATIENCE=NEW_PATIENCE
EPOCHS=previous_config['epochs'] if existing_phase.exists() else previous_config['epochs']+EXTRA_EPOCHS
SAVE_EVERY_UPDATES=25
ALLOW_CODE_FIX_RESUME=True
AUTO_RESUME_DISCOVERY=False
K2_WHEEL_URL='https://huggingface.co/csukuangfj/k2/resolve/f9222235425ce5eee567edda570c0abac4607ff0/ubuntu-cuda/k2-1.24.4.dev20250715+cuda12.4.torch2.5.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl#sha256=79e62f29bd0964f3919fe224385fb2e29e3d3513a4085b65bdd259475411722b'
print('CONTINUATION INPUT:',RESUME_DIR,'| LR:',LEARNING_RATE,'| extra epochs:',EXTRA_EPOCHS,'| patience:',PATIENCE)


## 2. GPU và Python riêng


In [ ]:
import sys, subprocess, os, json, shutil, time
WORK_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
ENV = dict(os.environ, PYTHONUNBUFFERED='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1',
           HF_HUB_DISABLE_XET='1', HF_XET_CHUNK_CACHE_SIZE_BYTES='0',
           HF_HOME=str(WORK_DIR/'hf'), UV_CACHE_DIR=str(WORK_DIR/'uv_cache'),
           UV_PYTHON_INSTALL_DIR=str(WORK_DIR/'managed_python'), OMP_NUM_THREADS='2')
# Kaggle may export UV_SYSTEM_PYTHON; explicit --python below owns this env.
ENV.pop('UV_SYSTEM_PYTHON',None)
def run(command, cwd=None):
    print('RUN:', ' '.join(str(x) for x in command),flush=True)
    with (OUTPUT_DIR/'run.log').open('a',encoding='utf-8') as log:
        process=subprocess.Popen([str(x) for x in command],cwd=cwd,env=ENV,
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace')
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        if process.wait()!=0:raise RuntimeError('Process failed; see run.log / failure.json in Output')
run(['nvidia-smi'])
run([sys.executable,'-m','pip','install','uv'])
VENV=WORK_DIR/'venv'
PYTHON=VENV/'bin/python'
if not PYTHON.exists():run([sys.executable,'-m','uv','venv','--python','3.11','--managed-python',VENV])

## 3. Cài dependencies


In [ ]:
run([sys.executable,'-m','uv','pip','install','--python',PYTHON,
     'torch==2.5.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu124'])
run([sys.executable,'-m','uv','pip','install','--python',PYTHON,
     'numpy==1.26.4','lhotse==1.32.0','soundfile==0.13.1','sentencepiece==0.2.1',
     'huggingface-hub==1.24.0','rapidfuzz==3.14.1','kaldialign==0.9.1',
     'pypinyin==0.50.0','tensorboard==2.19.0'])
run([sys.executable,'-m','uv','pip','install','--python',PYTHON,'--no-deps',K2_WHEEL_URL])
run([PYTHON,'-c',
     'import torch,torchaudio,k2,lhotse; from importlib.metadata import version; '
     'assert torch.cuda.is_available(), "Enable Kaggle GPU"; '
     'assert torch.__version__.startswith("2.5.1") and torch.version.cuda=="12.4"; '
     'assert k2.with_cuda, "k2 CPU wheel is not sufficient"; '
     'print("CUDA imports OK:",torch.cuda.get_device_name(0),torch.__version__,version("k2"))'])

## 4. Khóa Icefall và kiểm tra import


In [ ]:
ICEFALL_ROOT=WORK_DIR/'icefall'
if not (ICEFALL_ROOT/'.git').exists():
    ICEFALL_ROOT.mkdir(exist_ok=True)
    run(['git','init',ICEFALL_ROOT])
    run(['git','remote','add','origin','https://github.com/k2-fsa/icefall.git'],cwd=ICEFALL_ROOT)
head=subprocess.run(['git','rev-parse','HEAD'],cwd=ICEFALL_ROOT,text=True,capture_output=True).stdout.strip()
if head!=ICEFALL_COMMIT:
    run(['git','fetch','--depth','1','origin',ICEFALL_COMMIT],cwd=ICEFALL_ROOT)
    run(['git','checkout','--detach','FETCH_HEAD'],cwd=ICEFALL_ROOT)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ICEFALL_ROOT,text=True).strip()==ICEFALL_COMMIT
run([PYTHON,'-c','import sys;sys.path[:0]=[sys.argv[1],sys.argv[2]];from model import AsrModel;from optim import ScaledAdam;print("ICEFALL_IMPORT_OK")',ICEFALL_ROOT/'egs/librispeech/ASR/zipformer',ICEFALL_ROOT])

## 5. Ghi trainer continuation


In [ ]:
SOURCES = {'hynt_finetune_data.py': '"""Prepare a new train-only ASR experiment, including the historical 991 rows.\n\nDoes not change the team\'s manifests or read public-test/environment records.\nConversation and duplicate PCM components stay in one partition. No speaker\nidentity is inferred from folders, transcripts, or acoustic similarity.\n"""\nimport argparse\nfrom collections import defaultdict\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport unicodedata\nimport wave\nimport zipfile\n\nENGLISH_TERMS = (\'youtube\',\'facebook\',\'zalo\',\'gmail\',\'google\',\'wifi\',\'bluetooth\',\n                 \'spotify\',\'netflix\',\'tiktok\',\'messenger\',\'zoom\',\'app\',\'email\',\n                 \'online\',\'offline\',\'chrome\',\'excel\',\'word\',\'powerpoint\')\n\n\ndef normalize(text):\n    text = unicodedata.normalize(\'NFC\', str(text)).lower()\n    return \' \'.join(\'\'.join(\' \' if unicodedata.category(c)[0] in \'PS\' else c for c in text).split())\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'): digest.update(block)\n    return digest.hexdigest()\n\n\ndef stable_hash(value):\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,\n                                     separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix+\'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\', encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef english_hint(text):\n    return bool(set(normalize(text).split()) & set(ENGLISH_TERMS))\n\n\nclass Source:\n    def __init__(self, path):\n        path = Path(path).resolve()\n        self.archive = None\n        if path.is_file() and path.suffix.lower()==\'.zip\':\n            self.archive = zipfile.ZipFile(path)\n            names = self.archive.namelist()\n            if len(names)!=len(set(names)): raise ValueError(\'Duplicate ZIP members\')\n            candidates = [n for n in names if PurePosixPath(n).parts[-2:]==(\'train\',\'train.jsonl\')\n                          and \'__MACOSX\' not in PurePosixPath(n).parts]\n            if len(candidates)!=1: raise ValueError(\'Need one train/train.jsonl\')\n            self.manifest = candidates[0]\n            self.train = PurePosixPath(self.manifest).parent\n        else:\n            candidates = [path/\'train/train.jsonl\', path/\'train+public_test/train/train.jsonl\',\n                          path/\'train.jsonl\']\n            candidates = [p for p in candidates if p.is_file() and p.parent.name==\'train\']\n            if len(candidates)!=1: raise FileNotFoundError(\'Expected train/train.jsonl in DATASET_ROOT\')\n            self.manifest = candidates[0]\n            self.train = self.manifest.parent\n        self.path = path\n\n    def read_manifest(self):\n        return (self.archive.read(self.manifest) if self.archive else self.manifest.read_bytes())\n\n    def audio(self, relative):\n        p = PurePosixPath(relative)\n        if (p.is_absolute() or \'..\' in p.parts or \'\\\\\' in relative or \':\' in relative\n                or not p.parts or p.parts[0]!=\'audio\' or p.suffix.lower()!=\'.wav\'):\n            raise ValueError(\'Audio must stay in train/audio\')\n        if self.archive: return self.archive.read(str(self.train/p))\n        target = (self.train/Path(*p.parts)).resolve()\n        if not target.is_relative_to(self.train.resolve()): raise ValueError(\'Audio escapes train\')\n        return target.read_bytes()\n\n    def close(self):\n        if self.archive: self.archive.close()\n\n\ndef split_rows(rows, valid_fraction=.1, seed=2026):\n    if not 0 < valid_fraction < 1: raise ValueError(\'valid_fraction must be between 0 and 1\')\n    parent = {r[\'conversation_id\']: r[\'conversation_id\'] for r in rows}\n    def find(x):\n        while parent[x]!=x:\n            parent[x] = parent[parent[x]]; x=parent[x]\n        return x\n    seen = {}\n    for r in rows:\n        if r[\'pcm_sha256\'] in seen:\n            a,b=find(r[\'conversation_id\']), find(seen[r[\'pcm_sha256\']])\n            parent[max(a,b)] = min(a,b)\n        else: seen[r[\'pcm_sha256\']]=r[\'conversation_id\']\n    groups = defaultdict(list)\n    for r in rows: groups[find(r[\'conversation_id\'])].append(r)\n    if len(groups)<2: raise ValueError(\'Need two independent conversation/PCM groups\')\n    keys = sorted(groups, key=lambda x: stable_hash([seed,x]))\n    target = max(1, round(len(rows)*valid_fraction))\n    chosen, count = set(), 0\n    for k in keys:\n        if count+len(groups[k])<=target and len(chosen)<len(groups)-1:\n            chosen.add(k); count+=len(groups[k])\n    if not chosen: chosen={min(keys,key=lambda k:abs(len(groups[k])-target))}\n    train,valid=[],[]\n    for k in keys:\n        (valid if k in chosen else train).extend(groups[k])\n    if not train or not valid: raise ValueError(\'Empty partition\')\n    for key in (\'conversation_id\',\'pcm_sha256\',\'audio_relative\'):\n        if {r[key] for r in train} & {r[key] for r in valid}:\n            raise ValueError(\'Cross-partition overlap: \'+key)\n    return sorted(train,key=lambda r:r[\'uid\']), sorted(valid,key=lambda r:r[\'uid\'])\n\n\ndef prepare(dataset, work, output, valid_fraction=.1, seed=2026, expected_audio=5036):\n    work, output = Path(work).resolve(), Path(output).resolve()\n    if work==output: raise ValueError(\'Use separate work and result directories\')\n    source = Source(dataset)\n    rows, seen, ids = [], {}, set()\n    content = source.read_manifest()\n    try:\n        for line_no,line in enumerate(content.decode(\'utf-8-sig\').splitlines(),1):\n            if not line.strip(): continue\n            record=json.loads(line)\n            sid=record[\'id\']\n            if not isinstance(sid,str) or sid in ids: raise ValueError(\'Invalid/duplicate conversation ID\')\n            ids.add(sid)\n            turns=[(\'final\',None,record)]+[(\'history\',i,t) for i,t in enumerate(record.get(\'history\',[]))\n                   if t.get(\'role\') in (\'user\',\'human\') and t.get(\'audio\')]\n            for kind,index,turn in turns:\n                relative=turn.get(\'audio\'); text=turn.get(\'audio_text\',turn.get(\'text\'))\n                if not isinstance(relative,str) or not isinstance(text,str) or not normalize(text):\n                    raise ValueError(\'Missing audio/transcript; repair source, do not drop rows\')\n                if relative in seen:\n                    raise ValueError(\'Repeated manifest audio path; resolve ownership before splitting\')\n                payload=source.audio(relative)\n                with wave.open(io.BytesIO(payload),\'rb\') as w:\n                    sr,frames=w.getframerate(),w.getnframes()\n                    if sr!=16000 or w.getnchannels()!=1 or w.getsampwidth()!=2 or w.getcomptype()!=\'NONE\':\n                        raise ValueError(\'Requires mono 16kHz PCM16 WAV\')\n                    pcm=w.readframes(frames)\n                    if len(pcm)!=frames*2 or frames<=0: raise ValueError(\'Invalid WAV payload\')\n                target=work/Path(*PurePosixPath(relative).parts)\n                if not target.is_relative_to(work): raise ValueError(\'Output audio escapes work\')\n                target.parent.mkdir(parents=True,exist_ok=True)\n                if not target.exists() or sha_file(target)!=hashlib.sha256(payload).hexdigest():\n                    target.write_bytes(payload)\n                row=dict(uid=stable_hash([sid,kind,index,relative])[:24], conversation_id=sid,\n                    manifest_line=line_no, kind=kind, history_index=index,\n                    audio_relative=relative, audio_path=str(target), duration_s=frames/sr,\n                    pcm_sha256=hashlib.sha256(pcm).hexdigest(),\n                    wav_sha256=hashlib.sha256(payload).hexdigest(), reference=text,\n                    training_text=normalize(text).upper(), english_term_hint=english_hint(text))\n                seen[relative]=row; rows.append(row)\n    finally: source.close()\n    if len(rows)!=expected_audio:\n        raise ValueError(f\'Expected {expected_audio} train audio, got {len(rows)}; no silent subset\')\n    train,valid=split_rows(rows,valid_fraction,seed)\n    identity=[{k:r[k] for k in (\'uid\',\'conversation_id\',\'audio_relative\',\'pcm_sha256\',\'reference\')}\n              for r in sorted(rows,key=lambda r:r[\'uid\'])]\n    fingerprint=stable_hash(dict(rows=identity,seed=seed,valid_fraction=valid_fraction,\n        train_ids=[r[\'uid\'] for r in train],valid_ids=[r[\'uid\'] for r in valid]))\n    output.mkdir(parents=True,exist_ok=True)\n    old=output/\'split_summary.json\'\n    if old.exists() and json.loads(old.read_text(encoding=\'utf-8\'))[\'split_fingerprint\']!=fingerprint:\n        raise ValueError(\'Existing split differs; use a new output directory\')\n    for name,part in [(\'train\',train),(\'valid\',valid)]:\n        (output/(name+\'.jsonl\')).write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in part),encoding=\'utf-8\')\n    summary=dict(total_audio=len(rows),total_conversations=len(ids),train_audio=len(train),valid_audio=len(valid),\n        train_conversations=len({r[\'conversation_id\'] for r in train}),valid_conversations=len({r[\'conversation_id\'] for r in valid}),\n        train_hours=sum(r[\'duration_s\'] for r in train)/3600,valid_hours=sum(r[\'duration_s\'] for r in valid)/3600,\n        unique_pcm=len({r[\'pcm_sha256\'] for r in rows}),seed=seed,valid_fraction_requested=valid_fraction,\n        split_fingerprint=fingerprint,manifest_sha256=hashlib.sha256(content).hexdigest(),\n        train_english_term_hints=sum(r[\'english_term_hint\'] for r in train),\n        valid_english_term_hints=sum(r[\'english_term_hint\'] for r in valid),\n        conversation_disjoint=True,pcm_disjoint=True,speaker_disjoint_verified=False,\n        includes_historical_991=True, independent_test=False,\n        protocol=\'New development train/valid split of all 5036 train WAVs; historical 991 reused by user request\',\n        source=str(Path(dataset).resolve()))\n    save_json(old,summary)\n    print(json.dumps({k:v for k,v in summary.items() if k!=\'source\'},ensure_ascii=False,indent=2),flush=True)\n    return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--dataset\',required=True); p.add_argument(\'--work\',required=True)\n    p.add_argument(\'--output\',required=True);p.add_argument(\'--valid-fraction\',type=float,default=.1)\n    p.add_argument(\'--seed\',type=int,default=2026)\n    a=p.parse_args();prepare(a.dataset,a.work,a.output,a.valid_fraction,a.seed)\n', 'hynt_finetune.py': '"""Single-GPU hynt fine-tuning: joint pruned RNN-T, CTC and CR-CTC.\n\nUses the official Icefall architecture/optimizer at a locked revision. The\ntraining checkpoint is epoch 11, not a claim of identity to the avg-2 ONNX run.\nAll train/validation hypotheses are actual runtime outputs, never gold replay.\n"""\nimport argparse\nimport csv\nimport gc\nimport hashlib\nimport json\nimport math\nimport logging\nfrom pathlib import Path, PosixPath\nimport random\nimport sys\nimport time\n\nfrom hynt_finetune_data import normalize, save_json, sha_file, stable_hash\n\nMODEL_ID = \'hynt/ZipFormer-150M-CR-CTC-RNNT-6000h\'\nREVISION = \'8c3d474d3d74cb247f4b8c7b0f8b48598ac8c1c6\'\nCHECKPOINT_SHA = \'93d5f14e015ce5e3f06e9fce4d107769dc89c2856a60422635863c78cdb22baf\'\nBPE_SHA = \'002894e7a82d80ffa5e25008ec8c5496159db804005e2103de96b01b4c13d445\'\nSELECTION_METRIC = \'normalized_exact_match_then_wer\'\n\n\ndef better_validation_score(em,wer,best_em,best_wer):\n    if not math.isfinite(em) or not 0<=em<=1 or not math.isfinite(wer) or wer<0:\n        raise ValueError(\'Invalid validation score\')\n    return best_em is None or em>best_em or (em==best_em and wer<best_wer)\n\n\ndef require_em_checkpoint(saved):\n    if saved.get(\'selection_metric\')!=SELECTION_METRIC or saved.get(\'best_em\') is None:\n        raise ValueError(\'Old WER-selected run cannot resume as EM-selected: historical best-EM weights are unknown. Use a new OUTPUT_DIR for a fresh run; keep the old checkpoints.\')\n\n\nclass NonfiniteTrainingLoss(FloatingPointError):\n    def __init__(self,details):\n        self.details=details\n        super().__init__(\'Nonfinite training loss: \'+json.dumps(details,ensure_ascii=False))\n\n\ndef recover_loss(attempts,on_failure):\n    """Retry the same row; never replace a failed loss by zero or advance it."""\n    failures=[]\n    for name,calculate in attempts:\n        try:return calculate(),name\n        except NonfiniteTrainingLoss as error:\n            failures.append(dict(strategy=name,**error.details))\n            on_failure(failures[-1])\n    raise NonfiniteTrainingLoss(dict(all_attempts_failed=failures))\n\n\ndef stable_ctc(torch,log_probs,input_lengths,targets,target_lengths,on_fallback=None):\n    """Log-domain native CTC, plus a differentiable CPU float64 fallback.\n\n    Only this auxiliary loss changes backend; GPU encoder/convolutions remain\n    enabled. Never zero an infinite loss or ignore an impossible alignment.\n    """\n    log_probs=log_probs.float().contiguous()  # T, N, C\n    targets=targets.to(device=\'cpu\',dtype=torch.long).contiguous()\n    lengths=input_lengths.to(device=\'cpu\',dtype=torch.long)\n    target_lengths=target_lengths.to(device=\'cpu\',dtype=torch.long)\n    if not torch.isfinite(log_probs).all():\n        raise NonfiniteTrainingLoss(dict(ctc_reason=\'nonfinite_head_log_probs\'))\n    chunks=torch.split(targets,target_lengths.tolist())\n    for index,sequence in enumerate(chunks):\n        required=len(sequence)+int((sequence[1:]==sequence[:-1]).sum())\n        if required>int(lengths[index]):\n            raise NonfiniteTrainingLoss(dict(ctc_reason=\'impossible_alignment\',required=required,\n                available=int(lengths[index]),tokens=len(sequence)))\n    with torch.autocast(log_probs.device.type,enabled=False),torch.backends.cudnn.flags(enabled=False):\n        loss=torch.nn.functional.ctc_loss(log_probs,targets,lengths,target_lengths,\n            blank=0,reduction=\'sum\',zero_infinity=False)\n    if torch.isfinite(loss):return loss\n    if on_fallback:on_fallback(dict(reason=\'native_ctc_nonfinite\',backend=\'cpu_float64\',\n        frames=int(log_probs.shape[0]),batch=int(log_probs.shape[1]),tokens=target_lengths.tolist()))\n    # .to() keeps autograd: CTC gradients flow back to the GPU head/encoder.\n    cpu_probs=log_probs.to(device=\'cpu\',dtype=torch.float64)\n    loss64=torch.nn.functional.ctc_loss(cpu_probs,targets,lengths,target_lengths,\n        blank=0,reduction=\'sum\',zero_infinity=False)\n    if not torch.isfinite(loss64):\n        raise NonfiniteTrainingLoss(dict(ctc_reason=\'native_and_cpu_float64_nonfinite\'))\n    return loss64.to(log_probs.device)\n\n\ndef ctc_log_probs_fp32(torch,head,encoder_out):\n    with torch.autocast(encoder_out.device.type,enabled=False):\n        return head(encoder_out.float())\n\n\ndef compatible_resume_metadata(resume,output,saved_identity,cfg,split_fingerprint):\n    # Only the explicit new-stage controls may differ. Architecture is checked\n    # against params by the caller, before loading model/optimizer state.\n    core=(\'selection_metric\',\'icefall_commit\',\'seed\',\'gradient_accumulation\',\n          \'english_extra_fraction\',\'cs_concat_probability\',\'noise_probability\',\n          \'hard_extra_fraction\',\'augment_train\')\n    prior=json.loads((Path(resume).parent/\'run_identity.json\').read_text(encoding=\'utf-8\'))\n    if (prior.get(\'identity\')!=saved_identity or prior.get(\'split_fingerprint\')!=split_fingerprint\n            or any(prior.get(\'config\',{}).get(k)!=cfg.get(k) for k in core)):\n        raise ValueError(\'Continuation model/split/method provenance differs\')\n    return prior\n\n\ndef construct_model(params):\n    import torch\n    from decoder import Decoder\n    from joiner import Joiner\n    from model import AsrModel\n    from scaling import ScheduledFloat\n    from subsampling import Conv2dSubsampling\n    from zipformer import Zipformer2\n    ints=lambda key:tuple(int(x) for x in str(params[key]).split(\',\'))\n    dims=ints(\'encoder_dim\')\n    embed=Conv2dSubsampling(in_channels=params[\'feature_dim\'],out_channels=dims[0],\n                           dropout=ScheduledFloat((0.,.3),(20000.,.1)))\n    names=(\'downsampling_factor\',\'num_encoder_layers\',\'encoder_dim\',\'encoder_unmasked_dim\',\n           \'query_head_dim\',\'pos_head_dim\',\'value_head_dim\',\'num_heads\',\'feedforward_dim\',\n           \'cnn_module_kernel\',\'chunk_size\',\'left_context_frames\')\n    encoder=Zipformer2(output_downsampling_factor=2,**{k:ints(k) for k in names},\n        pos_dim=params[\'pos_dim\'],dropout=ScheduledFloat((0.,.3),(20000.,.1)),\n        warmup_batches=4000.,causal=params[\'causal\'])\n    decoder=Decoder(vocab_size=params[\'vocab_size\'],decoder_dim=params[\'decoder_dim\'],\n                    blank_id=params[\'blank_id\'],context_size=params[\'context_size\'])\n    joiner=Joiner(encoder_dim=max(dims),decoder_dim=params[\'decoder_dim\'],\n                  joiner_dim=params[\'joiner_dim\'],vocab_size=params[\'vocab_size\'])\n    class StableCTCModel(AsrModel):\n        # Override only the auxiliary head computation; all state_dict keys,\n        # parameter shapes, the transducer objective and decoding stay intact.\n        def forward_ctc(self,encoder_out,encoder_out_lens,targets,target_lengths):\n            lp=ctc_log_probs_fp32(torch,self.ctc_output,encoder_out)\n            return stable_ctc(torch,lp.permute(1,0,2),encoder_out_lens,targets,target_lengths,\n                getattr(self,\'ctc_fallback_callback\',None))\n        def forward_cr_ctc(self,encoder_out,encoder_out_lens,targets,target_lengths):\n            lp=ctc_log_probs_fp32(torch,self.ctc_output,encoder_out)\n            ctc=stable_ctc(torch,lp.permute(1,0,2),encoder_out_lens,targets,target_lengths,\n                getattr(self,\'ctc_fallback_callback\',None))\n            with torch.autocast(lp.device.type,enabled=False):\n                opposite=torch.roll(lp.detach(),lp.shape[0]//2,dims=0)\n                kl=torch.nn.functional.kl_div(lp,opposite,reduction=\'none\',log_target=True)\n                mask=torch.arange(lp.shape[1],device=lp.device)[None,:]>=encoder_out_lens[:,None]\n                cr=kl.masked_fill(mask.unsqueeze(-1),0.).sum()\n            return ctc,cr\n    return StableCTCModel(encoder_embed=embed,encoder=encoder,decoder=decoder,joiner=joiner,\n        encoder_dim=max(dims),decoder_dim=params[\'decoder_dim\'],vocab_size=params[\'vocab_size\'],\n        use_transducer=True,use_ctc=True,use_attention_decoder=False)\n\n\ndef read_jsonl(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\n\ndef atomic_torch_save(torch, data, path):\n    path=Path(path);temp=path.with_suffix(\'.tmp\')\n    torch.save(data,temp);temp.replace(path)\n\n\ndef check_finite_weights(torch,model):\n    checks=[torch.isfinite(p).all() for p in model.parameters()]\n    if checks and not torch.stack(checks).all().item():\n        raise FloatingPointError(\'Nonfinite model weights; previous last.pt/best.pt are kept untouched\')\n\n\ndef training_snapshot(torch,model,optimizer,scaler,identity,params,**progress):\n    return dict(identity=identity,model=model.state_dict(),optimizer=optimizer.state_dict(),\n        scaler=scaler.state_dict(),params=params,torch_rng=torch.get_rng_state(),\n        cuda_rng=torch.cuda.get_rng_state_all(),python_rng=random.getstate(),**progress)\n\n\ndef restore_training_snapshot(torch,saved,model,optimizer,scaler,identity):\n    if saved[\'identity\']!=identity:raise ValueError(\'Resume checkpoint belongs to another config/split/source\')\n    model.load_state_dict(saved[\'model\'],strict=True)\n    optimizer.load_state_dict(saved[\'optimizer\']);scaler.load_state_dict(saved[\'scaler\'])\n    torch.set_rng_state(saved[\'torch_rng\'])\n    # Training uses cuda:0 only; an unused GPU\'s saved RNG must not prevent\n    # resuming a T4x2 session on a single-GPU session.\n    torch.cuda.set_rng_state_all(saved[\'cuda_rng\'][:torch.cuda.device_count()])\n    random.setstate(saved[\'python_rng\'])\n    result={k:saved[k] for k in (\'global_step\',\'epoch\',\'cursor\',\'best_wer\',\'bad_epochs\')}\n    if \'best_em\' in saved:result[\'best_em\']=saved[\'best_em\']\n    return result\n\n\ndef set_batch_count(model, count):\n    for name,module in model.named_modules():\n        if hasattr(module,\'batch_count\'):module.batch_count=float(count)\n        if hasattr(module,\'name\'):module.name=name\n\n\ndef epoch_order(rows, seed, epoch, extra_english=.5):\n    rng=random.Random(seed+epoch*100003)\n    order=list(range(len(rows)))\n    english=[i for i,r in enumerate(rows) if r[\'english_term_hint\']]\n    if english: order+=rng.choices(english,k=round(len(english)*extra_english))\n    rng.shuffle(order)\n    return order\n\n\ndef main(config_path):\n    import importlib.metadata\n    import numpy as np\n    import soundfile as sf\n    import torch\n    import torchaudio\n    import k2\n    from huggingface_hub import hf_hub_download\n    from lhotse import Fbank,FbankConfig\n    from lhotse.dataset import SpecAugment\n    from rapidfuzz.distance import Levenshtein\n    import sentencepiece as spm\n\n    cfg=json.loads(Path(config_path).read_text(encoding=\'utf-8\'))\n    out,work=Path(cfg[\'output\']),Path(cfg[\'work\'])\n    out.mkdir(parents=True,exist_ok=True);work.mkdir(parents=True,exist_ok=True)\n    logging.basicConfig(level=logging.WARNING)\n    detail_log=logging.FileHandler(out/\'optimizer_warnings.log\',encoding=\'utf-8\')\n    logging.getLogger().addHandler(detail_log)\n    class OptimizerConsoleFilter(logging.Filter):\n        def filter(self,record):\n            return not record.getMessage().startswith((\'Clipping_scale=\',\'Scaling gradients by \',\n                \'Parameter dominating tot_sumsq \',\'Parameters with most larger-than-usual grads\'))\n    for handler in logging.getLogger().handlers:\n        if isinstance(handler,logging.StreamHandler) and not isinstance(handler,logging.FileHandler):\n            handler.addFilter(OptimizerConsoleFilter())\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable Kaggle GPU T4/P100; CUDA is required\')\n    if \'2.5.1\' not in torch.__version__ or torch.version.cuda!=\'12.4\':\n        raise RuntimeError(\'Use isolated notebook runtime torch 2.5.1+cu124\')\n    torch.set_num_threads(2)\n    torch.manual_seed(cfg[\'seed\']);torch.cuda.manual_seed_all(cfg[\'seed\'])\n    random.seed(cfg[\'seed\']);np.random.seed(cfg[\'seed\'])\n    recipe=Path(cfg[\'icefall_root\'])/\'egs/librispeech/ASR/zipformer\'\n    sys.path[:0]=[str(recipe),cfg[\'icefall_root\']]\n    from optim import ScaledAdam\n\n    train,valid=read_jsonl(out/\'train.jsonl\'),read_jsonl(out/\'valid.jsonl\')\n    split=json.loads((out/\'split_summary.json\').read_text(encoding=\'utf-8\'))\n    if len(train)+len(valid)!=5036:raise ValueError(\'Expected all 5036 audio in the new split\')\n    for key in (\'conversation_id\',\'pcm_sha256\'):\n        if {r[key] for r in train}&{r[key] for r in valid}:raise ValueError(\'Split leakage: \'+key)\n    identity=stable_hash(dict(split=split[\'split_fingerprint\'],\n        settings={k:v for k,v in cfg.items() if k not in (\'max_hours\',\'resume\')},\n        worker=sha_file(__file__),data_worker=sha_file(Path(__file__).with_name(\'hynt_finetune_data.py\'))))\n    run_identity=out/\'run_identity.json\'\n    if cfg.get(\'start_fresh\') and any((out/name).exists() for name in (\'last.pt\',\'best.pt\')):\n        raise FileExistsError(\'Fresh run requested but OUTPUT_DIR already has checkpoints. Choose a new OUTPUT_DIR; previous checkpoints were not overwritten.\')\n    resume=cfg.get(\'resume\') or (str(out/\'last.pt\') if (out/\'last.pt\').exists() else None)\n    prior_identity=json.loads(run_identity.read_text()) if run_identity.exists() else None\n    if prior_identity and prior_identity[\'identity\']!=identity and not (resume and cfg.get(\'allow_code_fix_resume\')):\n        raise ValueError(\'Config/source/split changed; use a new OUTPUT_DIR instead of mixing runs\')\n    if prior_identity and prior_identity[\'identity\']==identity and not cfg.get(\'resume\') and (out/\'state.json\').exists():\n        if json.loads((out/\'state.json\').read_text())[\'state\']==\'completed\':\n            print(\'This run is completed; use a new output directory or an explicit resume path.\',flush=True)\n            return\n    device=torch.device(\'cuda:0\')\n    save_json(out/\'runtime.json\',dict(python=sys.version,gpu=torch.cuda.get_device_name(0),\n        gpu_count=torch.cuda.device_count(),used_gpu=0,torch=torch.__version__,cuda=torch.version.cuda,\n        versions={d.metadata[\'Name\']:d.version for d in importlib.metadata.distributions()},\n        icefall_commit=cfg[\'icefall_commit\'],model_revision=REVISION,\n        baseline=\'pretrained.pt model weights; not guaranteed identical to epoch-11-avg-2 ONNX\',\n        decoding=\'RNN-T greedy, max_symbols_per_frame=1, no LM/hotwords\',fp16=True))\n    print(\'GPU:\',torch.cuda.get_device_name(0),\'| train/valid:\',len(train),len(valid),flush=True)\n    bpe=Path(hf_hub_download(MODEL_ID,\'bpe.model\',revision=REVISION,token=False))\n    pretrained=Path(hf_hub_download(MODEL_ID,\'pretrained.pt\',revision=REVISION,token=False))\n    for path,expected in [(bpe,BPE_SHA),(pretrained,CHECKPOINT_SHA)]:\n        if sha_file(path)!=expected:raise ValueError(\'Publisher checkpoint/tokenizer SHA256 mismatch\')\n    sp=spm.SentencePieceProcessor(model_file=str(bpe))\n    (out/\'bpe.model\').write_bytes(bpe.read_bytes())\n    # Only the standard pathlib class needed by this publisher\'s dict checkpoint.\n    # Never fall back to arbitrary/unrestricted pickle loading.\n    with torch.serialization.safe_globals([PosixPath]):\n        checkpoint=torch.load(pretrained,map_location=\'cpu\',weights_only=True)\n    keys=(\'feature_dim\',\'num_encoder_layers\',\'downsampling_factor\',\'feedforward_dim\',\'num_heads\',\n        \'encoder_dim\',\'query_head_dim\',\'value_head_dim\',\'pos_head_dim\',\'pos_dim\',\'encoder_unmasked_dim\',\n        \'cnn_module_kernel\',\'decoder_dim\',\'joiner_dim\',\'causal\',\'chunk_size\',\'left_context_frames\',\n        \'context_size\',\'blank_id\',\'vocab_size\')\n    params={k:checkpoint[k] for k in keys}\n    source_batch_count=int(checkpoint[\'batch_idx_train\'])\n    if params[\'vocab_size\']!=sp.vocab_size() or params[\'blank_id\']!=sp.piece_to_id(\'<blk>\'):\n        raise ValueError(\'Checkpoint/BPE vocabulary IDs differ\')\n    weights=checkpoint[\'model\'];del checkpoint;gc.collect()\n    model=construct_model(params)\n    model.load_state_dict(weights,strict=True)\n    del weights;gc.collect()\n    count=sum(p.numel() for p in model.parameters())\n    if count>=2_000_000_000:raise ValueError(\'ASR parameters must be below 2B\')\n    model.unk_id=sp.unk_id();model.to(device)\n    def ctc_fallback_report(event):\n        event={**getattr(model,\'ctc_context\',{}),**event}\n        with (out/\'ctc_backend_recovery.jsonl\').open(\'a\',encoding=\'utf-8\') as stream:\n            stream.write(json.dumps(event)+\'\\n\')\n        print(\'CTC BACKEND RECOVERY:\',event,flush=True)\n    model.ctc_fallback_callback=ctc_fallback_report\n    save_json(out/\'model_config.json\',dict(params=params,parameters=count,strict_load=True,\n        source_checkpoint_sha256=CHECKPOINT_SHA,bpe_sha256=BPE_SHA,source_batch_count=source_batch_count,\n        ctc_backend=\'FP32 head/CR + native log-domain CTC; differentiable CPU float64 fallback; zero_infinity=False\'))\n    print(\'Strict checkpoint load OK | parameters:\',count,flush=True)\n\n    # Never silently feed <unk>, blank or EOS as reference targets.\n    target_ids={}\n    unsupported=[]\n    for row in train+valid:\n        ids=sp.encode(row[\'training_text\'],out_type=int)\n        if not ids or set(ids)&{sp.unk_id(),params[\'blank_id\'],sp.bos_id(),sp.eos_id()}:\n            unsupported.append(dict(uid=row[\'uid\'],unknown_tokens=ids.count(sp.unk_id())))\n        target_ids[row[\'uid\']]=ids\n    save_json(out/\'tokenizer_audit.json\',dict(rows=len(target_ids),unsupported_rows=unsupported,\n        text_policy=\'NFC, lower, punctuation/symbol to space, uppercase for original BPE; digits/accents preserved\'))\n    if unsupported:raise ValueError(\'Unsupported training labels; see tokenizer_audit.json; rows not dropped\')\n\n    fbank=Fbank(FbankConfig(num_filters=80,dither=0.0,device=\'cpu\'))\n    save_json(out/\'fbank_config.json\',fbank.to_dict())\n    english=[r for r in train if r[\'english_term_hint\']]\n    def features(row,training=False,epoch=0,position=0):\n        cache=work/\'valid_features\'/(row[\'uid\']+\'.npy\')\n        if not training and cache.exists():return torch.from_numpy(np.load(cache))\n        audio,sr=sf.read(row[\'audio_path\'],dtype=\'float32\')\n        if sr!=16000 or audio.ndim!=1:raise ValueError(\'Unexpected audio format\')\n        text=row[\'training_text\']\n        rng=np.random.default_rng(int(stable_hash([cfg[\'seed\'],epoch,position,row[\'uid\']])[:16],16))\n        if training:\n            # Train-only concatenation of complete aligned utterances. This is\n            # inspired by CS mixing, not a reproduction of phrase-level mixing.\n            if english and rng.random()<cfg[\'cs_concat_probability\']:\n                other=english[int(rng.integers(len(english)))]\n                if other[\'uid\']!=row[\'uid\'] and row[\'duration_s\']+other[\'duration_s\']+.1<=24:\n                    second,sr2=sf.read(other[\'audio_path\'],dtype=\'float32\')\n                    if sr2!=16000 or second.ndim!=1:raise ValueError(\'Invalid concatenation audio\')\n                    audio=np.concatenate([audio,np.zeros(1600,dtype=\'float32\'),second])\n                    text+=\' \'+other[\'training_text\']\n            speed=float(rng.choice([.9,1.,1.1]))\n            if speed!=1.:\n                audio=torchaudio.functional.resample(torch.from_numpy(audio),int(16000*speed),16000).numpy()\n            if rng.random()<cfg[\'noise_probability\']:\n                rms=float(np.sqrt(np.mean(audio**2)+1e-12))\n                snr=float(rng.uniform(15,25))\n                audio=audio+rng.normal(0,rms/(10**(snr/20)),audio.shape).astype(\'float32\')\n                peak=float(np.max(np.abs(audio)))\n                if peak>1:audio=audio/peak\n        feat=fbank.extract(audio,16000).astype(\'float32\')\n        if not training:\n            cache.parent.mkdir(parents=True,exist_ok=True);np.save(cache,feat)\n            return torch.from_numpy(feat)\n        return torch.from_numpy(feat),sp.encode(text,out_type=int)\n\n    def greedy(encoder_out):\n        # Initial history and one-symbol-per-frame policy follow Icefall\'s\n        # pruned_transducer_stateless2/beam_search.py (Apache-2.0).\n        ctx=params[\'context_size\'];blank=params[\'blank_id\']\n        history=[blank]*ctx;result=[]\n        decoder_input=torch.tensor([[-1]*(ctx-1)+[blank]],device=device,dtype=torch.int64)\n        dec=model.joiner.decoder_proj(model.decoder(decoder_input,need_pad=False))\n        enc=model.joiner.encoder_proj(encoder_out)\n        for t in range(enc.shape[1]):\n            logits=model.joiner(enc[:,t:t+1,:].unsqueeze(2),dec.unsqueeze(1),project_input=False)\n            token=int(logits.argmax().item())\n            if token not in (blank,sp.unk_id()):\n                result.append(token);history.append(token)\n                decoder_input=torch.tensor([history[-ctx:]],device=device,dtype=torch.int64)\n                dec=model.joiner.decoder_proj(model.decoder(decoder_input,need_pad=False))\n        return result\n\n    def evaluate(name):\n        model.eval();set_batch_count(model,source_batch_count+global_step)\n        totals=dict(n=0,n_ok=0,n_error=0,word_edits=0,n_words=0,char_edits=0,n_chars=0,exact_normalized=0)\n        buckets={};predictions=out/(name+\'_predictions.jsonl\')\n        with predictions.open(\'w\',encoding=\'utf-8\') as stream,torch.inference_mode():\n            for i,row in enumerate(valid):\n                feat=features(row).unsqueeze(0).to(device)\n                lengths=torch.tensor([feat.shape[1]],device=device,dtype=torch.int64)\n                with torch.autocast(\'cuda\',dtype=torch.float16):\n                    enc,lens=model.forward_encoder(feat,lengths)\n                    hypothesis=sp.decode(greedy(enc[:,:int(lens[0]),:]))\n                ref,hyp=normalize(row[\'reference\']),normalize(hypothesis)\n                values=dict(n=1,n_ok=1,n_error=0,word_edits=Levenshtein.distance(ref.split(),hyp.split()),\n                    n_words=len(ref.split()),char_edits=Levenshtein.distance(ref.replace(\' \',\'\'),hyp.replace(\' \',\'\')),\n                    n_chars=len(ref.replace(\' \',\'\')),exact_normalized=int(ref==hyp))\n                record=dict(uid=row[\'uid\'],conversation_id=row[\'conversation_id\'],status=\'ok\',\n                    reference=row[\'reference\'],hypothesis=hypothesis,english_term_hint=row[\'english_term_hint\'],**values)\n                stream.write(json.dumps(record,ensure_ascii=False)+\'\\n\')\n                for bucket in [\'all\',row[\'kind\']]+([\'english_term_hint\'] if row[\'english_term_hint\'] else []):\n                    b=buckets.setdefault(bucket,{k:0 for k in totals})\n                    for k,v in values.items():b[k]+=v\n                if (i+1)%50==0:print(name,\'valid\',i+1,\'/\',len(valid),flush=True)\n        for b in buckets.values():\n            b[\'wer\']=b[\'word_edits\']/b[\'n_words\'];b[\'cer\']=b[\'char_edits\']/b[\'n_chars\']\n            b[\'exact_match\']=b[\'exact_normalized\']/b[\'n\']\n            b[\'ser\']=1-b[\'exact_match\']\n        save_json(out/(name+\'_metrics.json\'),dict(buckets=buckets,epoch=current_epoch,global_step=global_step,\n            selection_metric=SELECTION_METRIC,\n            split_fingerprint=split[\'split_fingerprint\'],independent_test=False,\n            note=\'English-term bucket is a fixed lexicon hint, not gold code-switch/dialect annotation\'))\n        score=buckets[\'all\']\n        print(name,\'EM:\',f"{score[\'exact_normalized\']}/{score[\'n\']}",\n            f"({100*score[\'exact_match\']:.3f}%)",\'WER:\',round(100*score[\'wer\'],3),\n            \'CER:\',round(100*score[\'cer\'],3),flush=True)\n        return score\n\n    optimizer=ScaledAdam(list(model.named_parameters()),lr=cfg[\'learning_rate\'],clipping_scale=2.0)\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=512.)\n    spec_aug=SpecAugment(time_warp_factor=0,num_frame_masks=2,features_mask_size=12,\n        num_feature_masks=2,frames_mask_size=20,max_frames_mask_fraction=.08)\n    def training_loss(feat, ids, use_amp=True, use_masks=True):\n        feat=feat.unsqueeze(0).to(device)\n        lens=torch.tensor([feat.shape[1]],device=device,dtype=torch.int64)\n        y=k2.RaggedTensor([ids])\n        with torch.autocast(\'cuda\',dtype=torch.float16,enabled=use_amp):\n            simple,pruned,ctc,_,cr=model(feat,lens,y,prune_range=5,am_scale=0.,lm_scale=.25,\n                use_cr_ctc=True,use_spec_aug=use_masks,spec_augment=spec_aug,\n                supervision_segments=torch.tensor([[0,0,feat.shape[1]]],dtype=torch.int32),time_warp_factor=0)\n            loss=(.5*simple+pruned+.1*ctc+.02*cr)/max(1,len(ids))\n        if not torch.isfinite(loss):\n            def value(x):\n                v=float(x.detach());return v if math.isfinite(v) else str(v)\n            raise NonfiniteTrainingLoss(dict(amp=use_amp,specaugment=use_masks,frames=int(feat.shape[1]),\n                tokens=len(ids),simple=value(simple),pruned=value(pruned),ctc=value(ctc),cr=value(cr)))\n        return loss\n    def row_loss(row,epoch,position,force_fp32=False):\n        model.ctc_context=dict(uid=row[\'uid\'],epoch=epoch,position=position,global_step=global_step)\n        feat,ids=features(row,True,epoch,position)\n        def report(details):\n            event=dict(uid=row[\'uid\'],epoch=epoch,position=position,global_step=global_step,**details)\n            with (out/\'loss_recovery.jsonl\').open(\'a\',encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(event,ensure_ascii=False)+\'\\n\')\n            print(\'LOSS RETRY:\',json.dumps(event,ensure_ascii=False),flush=True)\n        attempts=[]\n        if not force_fp32:attempts.append((\'fp16_augmented\',lambda:training_loss(feat,ids)))\n        attempts.extend([(\'fp32_augmented\',lambda:training_loss(feat,ids,False)),\n            (\'fp32_original\',lambda:training_loss(features(row),target_ids[row[\'uid\']],False,False))])\n        loss,strategy=recover_loss(attempts,report)\n        if strategy!=\'fp16_augmented\':\n            print(\'LOSS RECOVERED:\',row[\'uid\'],strategy,flush=True)\n        return loss\n    global_step,current_epoch,cursor,best_wer,bad_epochs=0,0,0,None,0\n    best_em=None\n    if resume:\n        saved=torch.load(resume,map_location=\'cpu\',weights_only=True)\n        require_em_checkpoint(saved)\n        accepted_identity=identity\n        if saved[\'identity\']!=identity:\n            if not cfg.get(\'allow_code_fix_resume\'):raise ValueError(\'Code-fix resume is disabled\')\n            if saved.get(\'params\')!=params:raise ValueError(\'Previous checkpoint architecture differs\')\n            prior=compatible_resume_metadata(resume,out,saved[\'identity\'],cfg,split[\'split_fingerprint\'])\n            accepted_identity=saved[\'identity\']\n            save_json(out/f\'previous_run_{accepted_identity}.json\',prior)\n            save_json(out/\'code_fix_resume.json\',dict(previous_identity=accepted_identity,new_identity=identity,\n                checkpoint=str(Path(resume).resolve()),checkpoint_sha256=sha_file(resume),\n                reason=\'Explicit continuation phase: new LR schedule and patience; model/split/method preserved\',\n                original_epoch=saved[\'epoch\'],original_cursor=saved[\'cursor\'],original_updates=saved[\'global_step\']))\n        progress=restore_training_snapshot(torch,saved,model,optimizer,scaler,accepted_identity)\n        global_step,current_epoch,cursor=progress[\'global_step\'],progress[\'epoch\'],progress[\'cursor\']\n        best_wer,bad_epochs=progress[\'best_wer\'],progress[\'bad_epochs\']\n        best_em=progress[\'best_em\']\n        del saved;gc.collect()\n        print(\'Resumed epoch\',current_epoch,\'cursor\',cursor,\'updates\',global_step,flush=True)\n        print(\'Checkpoint verified: preserving best.pt; baseline is not rerun.\',flush=True)\n    # A new phase resets patience once and starts a separate cosine LR schedule.\n    # Resuming an interrupted phase preserves its original boundaries and patience.\n    prior_plan=Path(resume).parent/\'continuation_plan.json\'\n    if prior_plan.is_file():\n        phase=json.loads(prior_plan.read_text(encoding=\'utf-8\'))\n        if phase[\'learning_rate\']!=cfg[\'learning_rate\'] or phase[\'patience\']!=cfg[\'patience\']:\n            raise ValueError(\'Keep the same LR/patience when resuming a continuation phase\')\n        if phase[\'extra_epochs\']!=cfg[\'extra_epochs\']:\n            raise ValueError(\'Keep the same extra_epochs when resuming a continuation phase\')\n        if not phase[\'start_epoch\']<=current_epoch<=phase[\'end_epoch\']+1:\n            raise ValueError(\'Checkpoint epoch outside continuation phase\')\n        if global_step<phase[\'start_step\']:\n            raise ValueError(\'Checkpoint predates continuation phase\')\n        print(\'RESUME CONTINUATION PHASE; patience counter preserved\',flush=True)\n    else:\n        if cursor!=0 or current_epoch<2:\n            raise ValueError(\'Start continuation from a completed epoch, not a partial epoch\')\n        phase=dict(start_epoch=current_epoch,end_epoch=current_epoch+cfg[\'extra_epochs\']-1,\n            start_step=global_step,extra_epochs=cfg[\'extra_epochs\'],\n            learning_rate=cfg[\'learning_rate\'],patience=cfg[\'patience\'],\n            origin_identity=accepted_identity)\n        bad_epochs=0\n        print(\'NEW CONTINUATION PHASE; reset patience once; preserve historical best\',flush=True)\n    save_json(out/\'continuation_plan.json\',phase)\n    print(\'CONTINUE epochs\',phase[\'start_epoch\'],\'to\',phase[\'end_epoch\'],\n          \'| current epoch\',current_epoch,\'| updates\',global_step,\n          \'| LR start\',phase[\'learning_rate\'],\'| patience\',phase[\'patience\'],flush=True)\n    save_json(run_identity,dict(identity=identity,config=cfg,split_fingerprint=split[\'split_fingerprint\']))\n\n    def save_last(epoch,next_cursor):\n        # Called only after an optimizer boundary: no partial accumulated grads.\n        check_finite_weights(torch,model)\n        atomic_torch_save(torch,training_snapshot(torch,model,optimizer,scaler,identity,params,\n            epoch=epoch,cursor=next_cursor,global_step=global_step,best_wer=best_wer,bad_epochs=bad_epochs,\n            best_em=best_em,selection_metric=SELECTION_METRIC),out/\'last.pt\')\n        save_json(out/\'state.json\',dict(state=\'checkpointed\',epoch=epoch,cursor=next_cursor,\n            updates=global_step,best_wer=best_wer,best_em=best_em,selection_metric=SELECTION_METRIC))\n\n    def save_best(score,epoch,source):\n        check_finite_weights(torch,model)\n        atomic_torch_save(torch,dict(model=model.state_dict(),params=params,identity=identity,\n            wer=score[\'wer\'],exact_match=score[\'exact_match\'],exact_normalized=score[\'exact_normalized\'],\n            n=score[\'n\'],selection_metric=SELECTION_METRIC,epoch=epoch,\n            global_step=global_step,source=source),out/\'best.pt\')\n\n    if best_wer is None:\n        raise ValueError(\'Continuation requires a trained EM-selected last.pt; refusing a fresh run\')\n        # Test the largest supported augmented feature shape, actual optimizer\n        # updates and the same checkpoint/restore path before full validation.\n        # Restore original weights/optimizer/RNG afterwards: baseline is untouched.\n        print(\'PREFLIGHT: long-shape loss/backward + 2 updates + checkpoint roundtrip\',flush=True)\n        model.ctc_context=dict(stage=\'preflight\')\n        original={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n        original_rng=(torch.get_rng_state(),torch.cuda.get_rng_state_all(),random.getstate())\n        model.train();set_batch_count(model,source_batch_count)\n        probe_row=max(train,key=lambda r:r[\'duration_s\'])\n        feat,ids=features(probe_row,True,0,0)\n        # Concat <=24 seconds then speed >=0.9 => <=26.67 seconds. Padding is a\n        # memory/shape stress fixture only, never part of training/quality metrics.\n        if feat.shape[0]<2700:feat=torch.nn.functional.pad(feat,(0,0,0,2700-feat.shape[0]))\n        successful=0\n        for attempt in range(8):\n            optimizer.zero_grad(set_to_none=True)\n            old_scale=scaler.get_scale()\n            for micro in range(2):\n                probe_loss,_=recover_loss([\n                    (\'fp16_stress\',lambda:training_loss(feat,ids)),\n                    (\'fp32_stress\',lambda:training_loss(feat,ids,False))],\n                    lambda details:print(\'PREFLIGHT LOSS RETRY:\',details,flush=True))\n                probe_loss=probe_loss/2\n                scaler.scale(probe_loss).backward()\n            scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:successful+=1\n            if successful==2:break\n        if successful!=2:raise FloatingPointError(\'CUDA preflight could not complete 2 finite optimizer updates\')\n        check_finite_weights(torch,model)\n        changed=any(not torch.equal(p.detach().cpu(),original[name]) for name,p in model.named_parameters())\n        if not changed:raise RuntimeError(\'Preflight optimizer did not change weights\')\n        optimizer.zero_grad(set_to_none=True)\n        probe_loss_value=float(probe_loss.detach());del probe_loss\n        probe_path=out/\'_preflight_roundtrip.pt\'\n        atomic_torch_save(torch,training_snapshot(torch,model,optimizer,scaler,identity,params,\n            epoch=1,cursor=0,global_step=successful,best_wer=None,bad_epochs=0),probe_path)\n        restored=torch.load(probe_path,map_location=\'cpu\',weights_only=True)\n        progress=restore_training_snapshot(torch,restored,model,optimizer,scaler,identity)\n        if progress[\'global_step\']!=successful:raise ValueError(\'Checkpoint update counter mismatch\')\n        model.load_state_dict(original,strict=True)\n        optimizer=ScaledAdam(list(model.named_parameters()),lr=cfg[\'learning_rate\'],clipping_scale=2.0)\n        scaler=torch.amp.GradScaler(\'cuda\',init_scale=512.)\n        torch.set_rng_state(original_rng[0]);torch.cuda.set_rng_state_all(original_rng[1]);random.setstate(original_rng[2])\n        save_json(out/\'training_probe.json\',dict(forward_backward=True,optimizer_updates=successful,\n            weights_changed=changed,checkpoint_save_load=True,baseline_weights_restored=True,\n            accumulated_micro_batches=2,feature_frames=int(feat.shape[0]),loss=probe_loss_value,gpu=torch.cuda.get_device_name(0),\n            peak_reserved_bytes=torch.cuda.max_memory_reserved()))\n        del restored,original,feat;probe_path.unlink();gc.collect();torch.cuda.empty_cache()\n        print(\'PREFLIGHT OK; original hynt weights restored. Starting baseline validation.\',flush=True)\n        score=evaluate(\'baseline\')\n        best_wer,best_em=score[\'wer\'],score[\'exact_match\']\n        save_best(score,0,\'baseline\')\n        save_last(1,0)\n        current_epoch=1\n    started=time.monotonic();accum=cfg[\'gradient_accumulation\']\n    # OOM/ABI/model mismatch errors propagate with the current checkpoint intact.\n    for epoch in range(max(1,current_epoch),phase[\'end_epoch\']+1):\n        current_epoch=epoch\n        order=epoch_order(train,cfg[\'seed\'],epoch,cfg[\'english_extra_fraction\'])\n        start=cursor\n        optimizer.zero_grad(set_to_none=True);model.train()\n        updates_per_epoch=math.ceil(len(order)/accum)\n        for offset in range(start,len(order),accum):\n            batch_indices=order[offset:offset+accum]\n            lr=cfg[\'learning_rate\']*(.5+.5*math.cos(math.pi*min(1.,max(0,global_step-phase[\'start_step\'])/(phase[\'extra_epochs\']*updates_per_epoch))))\n            for group in optimizer.param_groups:group[\'lr\']=max(lr,cfg[\'learning_rate\']*.1)\n            for attempt in range(8):\n                loss_value=0.\n                for micro,index in enumerate(batch_indices):\n                    row=train[index]\n                    set_batch_count(model,source_batch_count+global_step)\n                    loss=row_loss(row,epoch,offset+micro,force_fp32=attempt>0)/len(batch_indices)\n                    scaler.scale(loss).backward();loss_value+=float(loss.detach())\n                old_scale=scaler.get_scale()\n                scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)\n                if scaler.get_scale()>=old_scale:\n                    global_step+=1;break\n                print(\'AMP overflow: retry same accumulation group at scale\',scaler.get_scale(),flush=True)\n            else:raise FloatingPointError(\'8 failed AMP attempts; group was not advanced/skipped\')\n            cursor=offset+len(batch_indices)\n            if global_step%10==0:\n                print(f\'epoch {epoch}/{phase["end_epoch"]} | audio {cursor}/{len(order)} | update {global_step} | loss {loss_value:.4f} | lr {lr:.6g}\',flush=True)\n            if global_step and global_step%cfg[\'save_every_updates\']==0:save_last(epoch,cursor)\n            if time.monotonic()-started>cfg[\'max_hours\']*3600:\n                save_last(epoch,cursor)\n                # Protect the resumable checkpoint before validation. A partial\n                # epoch still needs a measured result for the report deadline.\n                name=f\'partial_epoch_{epoch:02}_step_{global_step:06}\'\n                print(\'Train time budget reached; checkpoint saved. Validating:\',name,flush=True)\n                score=evaluate(name)\n                if better_validation_score(score[\'exact_match\'],score[\'wer\'],best_em,best_wer):\n                    best_wer,best_em=score[\'wer\'],score[\'exact_match\'];bad_epochs=0\n                    save_best(score,epoch,\'fine_tuned_partial_epoch\')\n                save_last(epoch,cursor)\n                save_json(out/\'state.json\',dict(state=\'paused_time_budget\',epoch=epoch,cursor=cursor,\n                    updates=global_step,best_wer=best_wer,best_em=best_em,selection_metric=SELECTION_METRIC,\n                    validation_run=name,partial_epoch=True,\n                    independent_test=False,peak_vram_bytes=torch.cuda.max_memory_reserved()))\n                return\n        score=evaluate(f\'epoch_{epoch:02}\')\n        if global_step==0:raise RuntimeError(\'No successful optimizer update; inspect training precision/scaler\')\n        if better_validation_score(score[\'exact_match\'],score[\'wer\'],best_em,best_wer):\n            best_wer,best_em=score[\'wer\'],score[\'exact_match\'];bad_epochs=0\n            save_best(score,epoch,\'fine_tuned\')\n        else:bad_epochs+=1\n        cursor=0;save_last(epoch+1,0)\n        if bad_epochs>=cfg[\'patience\']:\n            print(\'Early stopping by EM, then WER; best.pt may still be the baseline.\',flush=True)\n            break\n    save_json(out/\'state.json\',dict(state=\'completed\',updates=global_step,best_wer=best_wer,\n        best_em=best_em,selection_metric=SELECTION_METRIC,\n        peak_vram_bytes=torch.cuda.max_memory_reserved(),independent_test=False))\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'config\')\n    args=p.parse_args()\n    try:main(args.config)\n    except Exception:\n        import traceback\n        cfg=json.loads(Path(args.config).read_text(encoding=\'utf-8\'))\n        save_json(Path(cfg[\'output\'])/\'failure.json\',dict(state=\'failed\',traceback=traceback.format_exc()))\n        raise\n'}

CODE_DIR=WORK_DIR/'code'
CODE_DIR.mkdir(exist_ok=True)
for name,source in SOURCES.items():
    (CODE_DIR/name).write_text(source,encoding='utf-8')
    old=OUTPUT_DIR/name
    if old.is_file() and old.read_text(encoding='utf-8')!=source:
        import hashlib
        digest=hashlib.sha256(old.read_bytes()).hexdigest()[:16]
        backup=OUTPUT_DIR/'previous_code'/digest
        backup.mkdir(parents=True,exist_ok=True)
        (backup/name).write_bytes(old.read_bytes())
    (OUTPUT_DIR/name).write_text(source,encoding='utf-8')


## 6. Dựng lại audio và kiểm tra split cũ


In [ ]:
candidates=[]
if DATASET_ROOT.exists():
    if DATASET_ROOT.is_file():candidates=[DATASET_ROOT]
    elif any(p.is_file() for p in [DATASET_ROOT/'train/train.jsonl',DATASET_ROOT/'train+public_test/train/train.jsonl']):
        candidates=[DATASET_ROOT]
    else:candidates=list(DATASET_ROOT.glob('train+public_test.zip'))
if not candidates:
    candidates=[p.parent.parent for p in DATASET_ROOT.rglob('train.jsonl')
                if p.parent.name=='train' and '__MACOSX' not in p.parts
                and not any(x.endswith('_environment') for x in p.parts)]
if not candidates:candidates=list(DATASET_ROOT.rglob('train+public_test.zip'))
candidates=list(dict.fromkeys(candidates))
if len(candidates)!=1:raise FileNotFoundError('Add the LifeMate dataset Input or set DATASET_ROOT to its unique train root/ZIP')
SOURCE_DATASET=candidates[0]
run([PYTHON,CODE_DIR/'hynt_finetune_data.py','--dataset',SOURCE_DATASET,
     '--work',WORK_DIR/'data','--output',OUTPUT_DIR,
     '--valid-fraction',str(VALID_FRACTION),'--seed',str(SEED)])
split=json.loads((OUTPUT_DIR/'split_summary.json').read_text(encoding='utf-8'))
print('Audio train/valid:',split['train_audio'],split['valid_audio'])
print('Overlap conversation/PCM: 0; speaker-disjoint chưa xác minh')

## 7. Giữ best cũ và cấu hình continuation


In [ ]:
# Recreated data uses current session audio paths; require the original split.
old_split=json.loads((RESUME_DIR/'split_summary.json').read_text(encoding='utf-8'))
if old_split['split_fingerprint']!=split['split_fingerprint']:
    raise ValueError('Split khác run cũ: không tiếp tục')
for field in ('icefall_commit','seed','gradient_accumulation','english_extra_fraction','cs_concat_probability','noise_probability'):
    if previous_config.get(field)!=dict(icefall_commit=ICEFALL_COMMIT,seed=SEED,
        gradient_accumulation=GRADIENT_ACCUMULATION,english_extra_fraction=ENGLISH_EXTRA_FRACTION,
        cs_concat_probability=CS_CONCAT_PROBABILITY,noise_probability=NOISE_PROBABILITY)[field]:
        raise ValueError('Method changed: '+field)
if (OUTPUT_DIR/'best.pt').exists() or (OUTPUT_DIR/'last.pt').exists():
    raise FileExistsError('Output đã có checkpoint. Dùng input là output mới nhất và OUTPUT_DIR khác để resume.')
if previous_config.get('hard_extra_fraction',0)>0 and not (RESUME_DIR/'hard_pool.json').is_file():
    raise FileNotFoundError('Nhánh Sherpa cần hard_pool.json để giữ nguyên replay train')
# Keep every old validation prediction, including earlier best epochs, for review.
for path in RESUME_DIR.iterdir():
    if path.is_file() and (path.name.endswith(('_metrics.json','_predictions.jsonl'))
                          or path.name in ('best.pt','hard_pool.json','continuation_plan.json')):
        shutil.copy2(path,OUTPUT_DIR/path.name)
provenance=OUTPUT_DIR/'previous_run_artifacts'/previous['identity']
provenance.mkdir(parents=True,exist_ok=True)
for name in ('run_identity.json','train_config.json','state.json','runtime.json','hynt_finetune.py','hynt_finetune_data.py'):
    path=RESUME_DIR/name
    if path.is_file():shutil.copy2(path,provenance/name)
config=dict(work=str(WORK_DIR),output=str(OUTPUT_DIR),icefall_root=str(ICEFALL_ROOT),
    selection_metric=SELECTION_METRIC,icefall_commit=ICEFALL_COMMIT,seed=SEED,epochs=EPOCHS,
    extra_epochs=EXTRA_EPOCHS,gradient_accumulation=GRADIENT_ACCUMULATION,learning_rate=LEARNING_RATE,
    patience=PATIENCE,max_hours=MAX_HOURS,save_every_updates=SAVE_EVERY_UPDATES,
    english_extra_fraction=ENGLISH_EXTRA_FRACTION,cs_concat_probability=CS_CONCAT_PROBABILITY,
    noise_probability=NOISE_PROBABILITY,resume=str(RESUME_DIR/'last.pt'),allow_code_fix_resume=True,start_fresh=False)
for field in ('hard_extra_fraction','augment_train'):
    if field in previous_config:config[field]=previous_config[field]
CONFIG_PATH=OUTPUT_DIR/'train_config.json'
CONFIG_PATH.write_text(json.dumps(config,indent=2),encoding='utf-8')
print('CONTINUE from last.pt; keep best.pt and all old validation files; source:',RESUME_DIR)


## 8. Train tiếp từ last.pt


In [ ]:
try:
    run([PYTHON,'-u',CODE_DIR/'hynt_finetune.py',CONFIG_PATH])
finally:
    for name in ('best.pt','last.pt','failure.json','loss_recovery.jsonl','ctc_backend_recovery.jsonl','code_fix_resume.json'):
        path=OUTPUT_DIR/name
        if path.exists():print('KEPT:',path,'| bytes:',path.stat().st_size)

## 9. Xem toàn bộ điểm cũ + mới và lưu output


In [ ]:
import hashlib
summary=[]
for path in sorted(OUTPUT_DIR.glob('*_metrics.json')):
    item=json.loads(path.read_text(encoding='utf-8'));row=item['buckets']['all']
    summary.append(dict(run=path.stem,n=row['n'],WER_pct=100*row['wer'],CER_pct=100*row['cer'],
                        exact=row['exact_normalized'],
                        EM_pct=100*row['exact_normalized']/row['n'],
                        SER_pct=100*(1-row['exact_normalized']/row['n']),epoch=item['epoch']))
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('State:',(OUTPUT_DIR/'state.json').read_text(encoding='utf-8'))
print('Checkpoints:',[p.name for p in OUTPUT_DIR.glob('*.pt')])
for name,source in SOURCES.items():
    (OUTPUT_DIR/name).write_text(source,encoding='utf-8')
def file_sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()
metadata={p.name:file_sha256(p) for p in OUTPUT_DIR.iterdir()
          if p.is_file() and p.name!='metadata_hashes.json'
          and p.suffix in ('.json','.jsonl','.py','.pt','.model')}
(OUTPUT_DIR/'metadata_hashes.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Kết quả ở:',OUTPUT_DIR,'| Chưa có test độc lập; không so thẳng với WER 15,34% trên 991 mẫu cũ.')